In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
%matplotlib inline
import pymc as pm
import numpy as np
import pandas as pd
from scipy.stats import norm, binom, bernoulli, lognorm, betabinom
from scipy.stats import beta as beta_dist
from scipy.special import expit as inv_logit
from scipy.special import logit ,logsumexp
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import arviz as az
import networkx as nx
import xarray as xr

import collections.abc
collections.Iterable = collections.abc.Iterable
from causalgraphicalmodels import CausalGraphicalModel
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
from matplotlib.patches import FancyArrowPatch

import warnings
warnings.filterwarnings('ignore')


%config InlineBackend.figure_format = 'retina'

plt.style.use(['seaborn-v0_8-darkgrid','seaborn-v0_8-colorblind'])

#### Code 12.1

In [ ]:
pbar = .5
theta = 5
sns.kdeplot(np.random.beta(pbar, theta,100))

#### Code  12.2

In [ ]:
d = pd.read_csv('data/UCBadmit.csv', sep = ';')
d['gid'] = np.where(d['applicant.gender'] == 'male', 0, 1)
dat = pd.DataFrame({
    'A': d.admit, 
    'N' : d.applications,
    'gid' : d.gid,
})

with pm.Model() as m12_1:
    phi = pm.Exponential('phi',1)
    theta = pm.Deterministic('theta', phi + 2.0)
    a = pm.Normal('a', 0, 1.5, shape = len(dat.gid.unique()))
    pbar = pm.Deterministic('pbar', pm.math.invlogit(a[dat.gid]))

    # beta-binomial parameters in pymc use different parameterization so this extra step is needed
    alpha= pm.Deterministic('alpha', pbar * theta) 
    beta = pm.Deterministic('beta', (1 - pbar) * theta)

    A = pm.BetaBinomial('A', n = dat.N, alpha = alpha, beta = beta, observed = dat.A)
    trace_12_1 = pm.sample(cores = 1, idata_kwargs = {'log_likelihood': True})


#### Code 12.3

In [ ]:
trace_12_1.posterior['da'] = trace_12_1.posterior['a'].sel(a_dim_0 = 0) - trace_12_1.posterior['a'].sel(a_dim_0 = 1)
pm.summary(trace_12_1, var_names = ['a','phi','theta','da'])

#### Code 12.4

In [ ]:
gid = 1
x = np.linspace(0,1,100)
# draw posterior mean beta distribution
alpha_mean = trace_12_1.posterior.alpha.values.mean()
beta_mean = trace_12_1.posterior.beta.values.mean()
plt.plot(x, beta_dist.pdf(x, alpha_mean, beta_mean), color = 'black');

samp_50 = trace_12_1.posterior.sel(a_dim_0 = gid).to_dataframe().reset_index().sample(50)

for i, row in samp_50.iterrows():
    plt.plot(x, beta_dist.pdf(x, row.alpha, row.beta), color = 'grey', alpha = 0.1)
plt.xlim(0,1);
plt.ylim(0,3);
plt.xlabel('probability admit');
plt.ylabel('Density');
plt.title('distribution of female admission rates');
    

#### Code 12.5

In [ ]:
d.head()

In [ ]:
post = trace_12_1.posterior.to_dataframe().reset_index() 
post['case'] = post.pbar_dim_0 + 1
d_grouped = d.groupby(['dept','gid'])[['admit','reject']].sum().reset_index()
d_grouped['case'] = d.index
d_grouped['rate'] = d_grouped['admit'] / (d_grouped['admit'] + d_grouped['reject'])

for i in post.case.unique():
    postcheck = post.loc[post.case == i]
    mean = postcheck.pbar.mean()
    hdi = postcheck.pbar.quantile([0.055, 0.945])
    pred_count = betabinom.rvs(d.applications.loc[d.index == i].values[0], postcheck.alpha, postcheck.beta) / d.applications.loc[d.index == i].values[0]
    pred_count_hdi = np.quantile(pred_count, [0.055, 0.945])
    plt.scatter(i, mean, marker = 'o', facecolor = 'none', edgecolor = 'black')
    plt.scatter(i, d_grouped['rate'].loc[d_grouped['case'] == i].values[0], marker = 'o', color = 'blue')
    plt.scatter(i, pred_count_hdi[0], color = 'black', marker = '+');
    plt.scatter(i, pred_count_hdi[1], color = 'black', marker = '+');
    plt.plot([i,i], hdi, color = 'black');
plt.xlabel('case');
plt.ylabel('A');
plt.title('Posterior validation check');
plt.ylim(0,1);
plt.xlim(.5,12.5);


#### Code 12.6

In [ ]:
d = pd.read_csv('data/Kline.csv', sep = ';')
d['P'] = scale.fit_transform(np.log(d[['population']]))
d['contact_id'] = np.where(d['contact'] == 'high', 1, 0)

dat2 = pd.DataFrame({
    'T': d.total_tools,
    'P': d.population,
    'cid': d.contact_id
})

with pm.Model() as m12_2:
    phi = pm.Exponential('phi', 1)
    g = pm.Exponential('g', 1)
    a = pm.Normal('a', 1,1, shape = len(dat2.cid.unique()))
    b = pm.Exponential('b', 1, shape = len(dat2.cid.unique()))
    lambdas = pm.Deterministic(
        'lambdas', 
        pm.math.exp(a[dat2.cid.values]) * dat2.P.values ** b[dat2.cid.values] / g
    )    
    T = pm.NegativeBinomial('T', mu=lambdas, alpha=phi, observed=dat2.T.values)
    trace_m12_2 = pm.sample(cores = 1, idata_kwargs={'log_likelihood': True})
    


#### Code 12.7

In [ ]:
prob_drink = .2
rate_work = 1

N = 365
drink = np.random.binomial(1, prob_drink, N)

y = (1-drink) * np.random.poisson(rate_work, N)


#### Code 12.8

In [ ]:
plt.hist(y, width=0.5, color='black', align= 'left' )

zeros_drink = np.sum(drink == 1)  
zeros_work = np.sum((y == 0) & (drink == 0))  


plt.bar(0, zeros_drink, bottom=zeros_work, width=0.5, color='blue', align = 'center')

plt.xlabel('manuscripts completed')
plt.ylabel('Frequency')


#### Code 12.9

In [ ]:
with pm.Model() as m12_3: 
    al = pm.Normal('al', 1, .5)
    ap = pm.Normal('ap', -1.5, 1)
    log_lambda = pm.math.exp(al)
    logit_p = pm.math.sigmoid(ap)
    y_obs = pm.ZeroInflatedPoisson('y_obs', mu = log_lambda, psi = logit_p, observed = y)
    trace_12_3 = pm.sample(cores = 1, idata_kwargs = {'log_likelihood': True})
pm.summary(trace_12_3)

#### Code 12.10

In [ ]:
print(inv_logit(trace_12_3.posterior.ap.values).mean())
print(np.exp(trace_12_3.posterior.al.values).mean())

#### Code 12.11

_This is some under the hood STAN stuff and is not relevant to how PyMC computes ZIPoisson_

#### Code 12.12

In [ ]:
d = pd.read_csv('data/Trolley.csv', sep = ';') 
d['id'] = d['id'].str.replace(';','-')


#### Code 12.13

In [ ]:
plt.hist(d.response, bins = np.arange(.5,8.5,1), rwidth = .1, color= 'black')
plt.xlim(0,8)
plt.xlabel('response')

#### Code 12.14

In [ ]:
pr_k = d['response'].value_counts(normalize=True).sort_index()
cum_pr_k = pr_k.cumsum()

plt.plot(np.arange(1,8),cum_pr_k, 'o-r', color = 'black', fillstyle ='none')
plt.ylim(0,1.05)


#### Code 12.15

In [ ]:
logit_pr_r = pd.DataFrame({'response': np.arange(7), 'logit_val': logit(cum_pr_k)}).reset_index(drop=True)

logit_d = logit_pr_r[~logit_pr_r['logit_val'].isna()]

#plt.plot(np.arange(1,8),logit_pr_r, 'o-r', color = 'black', fillstyle ='none')

logit_d

#### Code 12.16-12.17

_Note: 12.17 is just the same model written in quap instead of Ulam._

In [ ]:
logit_d['response'].values

In [ ]:
d.head()

In [ ]:
with pm.Model() as m12_4:
    cutpoints = pm.Normal('cutpoints', 0, 1.5, shape=6,
                          initval=np.array([-1.5, -1.0, -0.5, 0, 0.5, 1.0]))
    R = pm.OrderedLogistic('R', eta=0, cutpoints=cutpoints, 
                           observed=d['response'].values - 1)
    trace_m12_4 = pm.sample(cores = 1,init="adapt_diag")

#### Code 12.18

In [ ]:
summary = pm.summary(trace_m12_4, var_names = ['cutpoints'])

#### Code 12.19

In [ ]:
round(inv_logit(summary['mean']),3)


#### Code 12.20

In [ ]:
trace_m12_4.posterior.cutpoints.values.reshape(-1,6).mean(axis=0)

In [ ]:
cutpoints = inv_logit(trace_m12_4.posterior.cutpoints.values.reshape(-1,6).mean(axis=0))

pk = np.concatenate([[cutpoints[0]], 
                     np.diff(cutpoints), 
                     [1 - cutpoints[-1]]])
print(np.round(pk, 2))

#### Code 12.21

In [ ]:
np.sum(pk * np.arange(1,8))

#### Code 12.22

In [ ]:
cutpoints = inv_logit(trace_m12_4.posterior.cutpoints.values.reshape(-1,6).mean(axis=0) -.5)

print(cutpoints )

pk = np.concatenate([[cutpoints[0]], 
                     np.diff(cutpoints), 
                     [1 - cutpoints[-1]]])
print(np.round(pk, 2))

#### Code 12.23

In [ ]:
np.sum(pk * np.arange(1,8))

#### Code 12.24

In [ ]:
dat = pd.DataFrame({
    'R': d.response.values - 1,
    'A': d.action,
    'I': d.intention,
    'C': d.contact
})

with pm.Model() as model:
    cutpoints = pm.Normal('cutpoints', 0, 1.5, shape=6,
                          initval=np.array([-1.5, -1.0, -0.5, 0, 0.5, 1.0]))
    bA = pm.Normal('bA', 0, .5)
    bI = pm.Normal('bI', 0, .5) 
    bC = pm.Normal('bC', 0, .5)
    bIA = pm.Normal('bIA', 0, .5)
    bIC = pm.Normal('bIC', 0, .5)
    BI = pm.Deterministic('BI', bI + bIA * dat.A + bIC * dat.C)
    phi  = pm.Deterministic('phi', bA * dat.A + bC * dat.C + BI * dat.I)
    R = pm.OrderedLogistic('R', eta=phi, cutpoints=cutpoints, 
                           observed=dat['R'].values)
    trace_m12_5 = pm.sample(cores = 1, init="adapt_diag")

pm.summary(trace_m12_5, var_names = ['bA', 'bI', 'bC', 'bIA', 'bIC'])


#### Code 12.25

In [ ]:
az.plot_forest(trace_m12_5, var_names = ['bIC', 'bIA', 'bC','bI','bA'], combined = True, figsize = (10, 8));

In [ ]:
plt.figure()
plt.xlabel('intention')
plt.ylabel('probability')
plt.xlim(0,1)
plt.ylim(0,1)
plt.xticks([0, 1])
plt.yticks([0, 0.5, 1]);


#### Code 12.27

In [ ]:
kA = 0 
kC = 0 
kI = [0,1] 
pdat = pd.DataFrame({'A': kA, 'C': kC, 'I': kI})

bA = trace_m12_5.posterior['bA'].values.flatten()
bC = trace_m12_5.posterior['bC'].values.flatten()
bI = trace_m12_5.posterior['bI'].values.flatten()

phi = (bA[:, None] * pdat['A'].values + 
       bC[:, None] * pdat['C'].values + 
       bI[:, None] * pdat['I'].values)
print(phi[:10])
(phis * kI_array.T)[:10]

#### Code 12.28

In [ ]:
cutpoints = trace_m12_5.posterior['cutpoints'].values.reshape(-1, 6)  
phis = trace_m12_5.posterior['bI'].values.flatten()  
sample_idx = np.random.choice(cutpoints.shape[0], 50, replace=False)

for s in sample_idx:
    cutpoint = cutpoints[s]
    phi = phis[s]
    
    kI_array = np.array(kI)[:, None]  
    pk = inv_logit(cutpoint - (phi * kI_array))  
    for j in range(6):
        plt.plot(kI, pk[:, j], color='black', alpha=0.1)

In [ ]:
kA = 0
kC = 1
kI = np.array([0, 1])

cutpoints = trace_m12_5.posterior['cutpoints'].values.reshape(-1, 6)  
bI = trace_m12_5.posterior['bI'].values.flatten()
bA = trace_m12_5.posterior['bA'].values.flatten()
bC = trace_m12_5.posterior['bC'].values.flatten()

n_samples = len(bI)

phi = bI[:, None] * kI + bA[:, None] * kA + bC[:, None] * kC  

cum_probs = inv_logit(cutpoints[:, None, :] - phi[:, :, None])

probs = np.zeros((n_samples, len(kI), 7))
probs[:, :, 0] = cum_probs[:, :, 0]
probs[:, :, 1:6] = np.diff(cum_probs, axis=2)
probs[:, :, 6] = 1 - cum_probs[:, :, 5]

probs_flat = probs.reshape(-1, 7)  
predictions = np.array([np.random.choice(7, p=p) for p in probs_flat])

plt.hist(predictions, bins=np.arange(0, 7.5, 1), 
        color = 'black', 
        width = 0.1,
        align = 'mid')
plt.xlabel('response')
plt.xticks(range(7), range(1,8))

print(probs.mean(axis=(0,1)))

#### Code 12.30

In [ ]:
d = pd.read_csv('data/Trolley.csv', sep=';')
labels, unique = pd.factorize(d.edu)
raw_dict = {u:l for l,u in enumerate(unique)}

#### Code 12.31

In [ ]:
edu_levels = [7,0,6,4,2,1,5,3] 
ordered_dict = {k: edu_levels.index(raw_dict.get(k)) for k in raw_dict.keys()}
d['edu_new'] = d.edu.map(ordered_dict)


#### Code 12.32

In [ ]:
np.random.seed(1806)
delta = np.random.dirichlet(np.repeat(2,7), 10)

#### Code 12.33

In [ ]:
h = 3

plt.figure()
plt.xlim(1,7)
plt.ylim(0,.4)
plt.xlabel('index')
plt.ylabel('probability')


for i in range(delta.shape[0]):
    np.where(i!= h, plt.plot(np.linspace(0,7,7), delta[i,], color = 'black', alpha = 0.1),
    plt.plot(np.linspace(0,7,7), delta[h,], color = 'black', lw = 2))


#### Code 12.34

In [ ]:
np.repeat(2,7)

In [ ]:
dat = pd.DataFrame({ 
    'R': d.response,
    'action': d.action,
    'intention': d.intention,
    'contact': d.contact,
    'E': d.edu_new,
})


with pm.Model() as m12_6:
    delta = pm.Dirichlet('delta', a=np.repeat(2,7))
    delta_j = pm.Deterministic('delta_j', pm.math.concatenate([[0], delta]))
    cumsum_delta = pm.Deterministic('cumsum_delta', pm.math.cumsum(delta_j))
    
    bA = pm.Normal('bA', 0, 1)
    bI = pm.Normal('bI', 0, 1)
    bC = pm.Normal('bC', 0, 1)
    bE = pm.Normal('bE', 0, 1)
    kappa = pm.Normal('kappa', 0, 1.5, shape=6, initval=np.linspace(-2, 2, 6))
    
    phi = pm.Deterministic('phi', 
        bE * cumsum_delta[dat.E.values] +  
        bA * dat.action.values + 
        bI * dat.intention.values + 
        bC * dat.contact.values
    )
    
    R = pm.OrderedLogistic('R', eta=phi, cutpoints=kappa, 
                           observed=dat.R.values - 1)
    trace_m12_6 = pm.sample(400, tune = 400,cores=1, init="adapt_diag")






#### Code 12.35

In [ ]:
pm.summary(trace_m12_6, var_names = ['bE', 'bC', 'bI', 'bA', 'delta'])

In [ ]:
trace_m12_6

In [ ]:
delta_labels = ["Elem","MidSch","SHS","HSG","SCol","Bach","Mast","Grad"]
trace_m12_6.posterior.assign_coords(delta_j_dim_0 = delta_labels)
delta_samples = trace_m12_6.posterior['delta_j'].values.reshape(-1, 8)
delta_df = pd.DataFrame(delta_samples, columns=delta_labels)

sns.pairplot(delta_df);

#### Code 12.37

In [ ]:
dat['edu_norm'] = scaler.fit_transform(d[['edu_new']])


with pm.Model() as m12_7:
    cutpoints = pm.Normal('cutpoints', 0, 1.5, shape=6,
                          initval=np.array([-1.5, -1.0, -0.5, 0, 0.5, 1.0]))
    bA = pm.Normal('bA', 0, 1)
    bI = pm.Normal('bI', 0, 1)
    bC = pm.Normal('bC', 0, 1)
    bE = pm.Normal('bE', 0, 1)
    mu = pm.Deterministic('mu', 
        bE * dat.edu_norm.values +  
        bA * dat.action.values + 
        bI * dat.intention.values + 
        bC * dat.contact.values
    )
    R = pm.OrderedLogistic('R', eta=mu, cutpoints=cutpoints, 
                           observed=dat.R.values - 1)
    trace_m12_7 = pm.sample(400, tune = 400,cores=1, init="adapt_diag")

pm.summary(trace_m12_7, var_names = ['bE', 'bC', 'bI', 'bA'])